In [0]:
# ============================================================
# NOTEBOOK 04 — EARLY ALERT SYSTEM
# Vermont School - Early Warning System V2
#
# Combines model predictions with T3 partial grades
# to generate 4 actionable alert categories
# ============================================================

SILVER  = "/Volumes/workspace/vermont/silver"
TRUSTED = "/Volumes/workspace/vermont/trusted"

PRED_PATH    = f"{SILVER}/predictions_25_26"
ALERT_OUTPUT = f"{SILVER}/early_alerts_25_26"

GROUPS = [
    'Science', 'I_and_S', 'Mathematics', 'English',
    'Lengua_Castellana', 'Mandarin', 'Financial_Maths',
    'ICT_STEM', 'Physical_Education', 'Research_Methodology'
]

WEIGHTS = {
    'T1': 0.30,
    'T2': 0.30,
    'T3': 0.40
}

import pandas as pd
import numpy as np

# Load predictions
df = spark.read.parquet(PRED_PATH).toPandas()
print(f"✓ Predictions loaded: {len(df)} students")
print(f"\nPredicted risk distribution:")
print(df['predicted_risk'].value_counts().to_string())

In [0]:
# CELDA 2 — Calculate minimum T3 grade needed per subject
# and cross with T3 partial

# T3 partial columns available
t3_cols = [f'{g}_T3' for g in GROUPS if f'{g}_T3' in df.columns]
t1_cols = [f'{g}_T1' for g in GROUPS if f'{g}_T1' in df.columns]
t2_cols = [f'{g}_T2' for g in GROUPS if f'{g}_T2' in df.columns]

print(f"T3 partial columns available: {len(t3_cols)}")

# For each subject calculate:
# 1. Minimum T3 needed to pass (≥4.0 cumulative)
# 2. Whether T3 partial is above/below that minimum
# 3. Whether passing is mathematically possible

results = []
for _, row in df.iterrows():
    rec = {
        'student_id':      row['student_id'],
        'grade':           row['grade'],
        'section_anon':    row['section_anon'],
        'predicted_risk':  row['predicted_risk'],
        'current_risk':    row['risk_level'],
        'confidence':      row['confidence'],
        'avg_cumulative':  row['avg_cumulative'],
        'min_cumulative':  row['min_cumulative'],
        'n_subjects_below_4': row['n_subjects_below_4'],
    }

    subjects_at_risk = []
    subjects_impossible = []
    subjects_recovering = []

    for g in GROUPS:
        t1 = row.get(f'{g}_T1', np.nan)
        t2 = row.get(f'{g}_T2', np.nan)
        t3 = row.get(f'{g}_T3', np.nan)

        if pd.isna(t1) or pd.isna(t2):
            continue

        # Minimum T3 needed to reach 4.0 cumulative
        # 4.0 = T1*0.30 + T2*0.30 + T3*0.40
        min_t3 = (4.0 - t1 * 0.30 - t2 * 0.30) / 0.40
        rec[f'{g}_min_T3_needed'] = round(min_t3, 2)

        if min_t3 > 7.0:
            # Mathematically impossible to pass
            subjects_impossible.append(g)
        elif not pd.isna(t3):
            if t3 < min_t3:
                subjects_at_risk.append(g)
            else:
                subjects_recovering.append(g)

    rec['subjects_impossible']  = ', '.join(subjects_impossible) if subjects_impossible else 'None'
    rec['n_impossible']         = len(subjects_impossible)
    rec['subjects_at_risk']     = ', '.join(subjects_at_risk) if subjects_at_risk else 'None'
    rec['n_at_risk_t3']         = len(subjects_at_risk)
    rec['subjects_recovering']  = ', '.join(subjects_recovering) if subjects_recovering else 'None'
    rec['n_recovering']         = len(subjects_recovering)
    results.append(rec)

df_alerts = pd.DataFrame(results)

print(f"\n── T3 Analysis ──")
print(f"Students with impossible subjects: {(df_alerts['n_impossible'] > 0).sum()}")
print(f"Students with subjects at risk in T3: {(df_alerts['n_at_risk_t3'] > 0).sum()}")
print(f"Students recovering in T3: {(df_alerts['n_recovering'] > 0).sum()}")
print(f"\nImpossible subject distribution:")
print(df_alerts[df_alerts['n_impossible']>0]['subjects_impossible']\
    .value_counts().head(10).to_string())

In [0]:
# Debug — verificar valores T3
print("T3 columns in df:")
for col in t3_cols:
    non_null = df[col].notna().sum()
    print(f"  {col}: {non_null} non-null values")

print(f"\nSample T3 values (first 5 rows):")
print(df[t3_cols[:3]].head(5).to_string())

In [0]:
# Debug — calcular min_t3 needed manualmente
print("Muestra de cálculos min_T3_needed:")
for g in ['Mathematics', 'Science', 'English']:
    t1_col = f'{g}_T1'
    t2_col = f'{g}_T2'
    t3_col = f'{g}_T3'
    
    if t1_col in df.columns and t2_col in df.columns:
        df[f'{g}_min_T3'] = (4.0 - df[t1_col]*0.30 - df[t2_col]*0.30) / 0.40
        df[f'{g}_gap'] = df[f'{g}_min_T3'] - df[t3_col]
        
        at_risk = (df[f'{g}_gap'] > 0).sum()
        impossible = (df[f'{g}_min_T3'] > 7.0).sum()
        
        print(f"\n  {g}:")
        print(f"    Min T3 needed avg: {df[f'{g}_min_T3'].mean():.2f}")
        print(f"    Min T3 needed max: {df[f'{g}_min_T3'].max():.2f}")
        print(f"    Students at risk in T3: {at_risk}")
        print(f"    Mathematically impossible: {impossible}")
        print(f"    T3 partial avg: {df[t3_col].mean():.2f}")

In [0]:
# Recargar con T1 y T2 desde trusted/predict_dataset
df_full = spark.read.parquet(
    f"{TRUSTED}/predict_dataset"
).toPandas()

# Verificar T1 y T2
print("T1/T2 columns available:")
for g in ['Mathematics', 'Science', 'English']:
    for t in ['T1', 'T2', 'T3']:
        col = f'{g}_{t}'
        if col in df_full.columns:
            non_null = df_full[col].notna().sum()
            print(f"  {col}: {non_null} non-null")

# Merge con predicciones
df_merged = df_full.merge(
    df[['student_id', 'predicted_risk', 'confidence']],
    on='student_id',
    how='left'
)
print(f"\n✓ Merged: {len(df_merged)} students")
print(f"Columns: {len(df_merged.columns)}")

In [0]:
# CELDA 3 — Alert categories: Model prediction + T3 partial reality

WEIGHTS = {'T1': 0.30, 'T2': 0.30, 'T3': 0.40}

results = []
for _, row in df_merged.iterrows():
    rec = {
        'student_id':         row['student_id'],
        'grade':              row['grade'],
        'section_anon':       row['section_anon'],
        'predicted_risk':     row.get('predicted_risk', 'unknown'),
        'current_risk':       row['risk_level'],
        'confidence':         row.get('confidence', 0),
        'avg_cumulative':     row['avg_cumulative'],
        'min_cumulative':     row['min_cumulative'],
        'n_subjects_below_4': row['n_subjects_below_4'],
    }

    # ── Per subject analysis ──
    subjects_detail = []
    n_impossible = 0
    n_at_risk_t3 = 0
    n_recovering = 0

    for g in GROUPS:
        t1 = row.get(f'{g}_T1', np.nan)
        t2 = row.get(f'{g}_T2', np.nan)
        t3 = row.get(f'{g}_T3', np.nan)

        if pd.isna(t1) or pd.isna(t2):
            continue

        # Minimum T3 needed to reach 4.0
        min_t3 = (4.0 - t1 * 0.30 - t2 * 0.30) / 0.40

        # Projected final grade if T3 partial = T3 final
        projected = t1*0.30 + t2*0.30 + (t3*0.40 if not pd.isna(t3) else 0)

        # T3 status
        if min_t3 > 7.0:
            t3_status = 'IMPOSSIBLE'
            n_impossible += 1
        elif pd.isna(t3):
            t3_status = 'PENDING'
        elif t3 < min_t3:
            t3_status = 'AT_RISK'
            n_at_risk_t3 += 1
        else:
            t3_status = 'RECOVERING'
            n_recovering += 1

        subjects_detail.append({
            'subject':    g,
            'T1':         round(t1, 2),
            'T2':         round(t2, 2),
            'T3_partial': round(t3, 2) if not pd.isna(t3) else None,
            'min_T3':     round(min_t3, 2),
            'projected':  round(projected, 2),
            'T3_status':  t3_status,
        })

        rec[f'{g}_T1']         = round(t1, 2)
        rec[f'{g}_T2']         = round(t2, 2)
        rec[f'{g}_T3_partial'] = round(t3, 2) if not pd.isna(t3) else np.nan
        rec[f'{g}_min_T3']     = round(min_t3, 2)
        rec[f'{g}_projected']  = round(projected, 2)
        rec[f'{g}_T3_status']  = t3_status

    rec['n_impossible']  = n_impossible
    rec['n_at_risk_t3']  = n_at_risk_t3
    rec['n_recovering']  = n_recovering

    # ── ALERT CATEGORY ──
    # Model dimension
    pred = rec['predicted_risk']
    conf = rec['confidence']

    # Reality dimension (T3 partial)
    # Is T3 confirming or contradicting model?
    current = rec['current_risk']

    if pred == 'critical' and (n_at_risk_t3 >= 3 or n_impossible >= 1):
        category = '🔴 CONFIRMED_CRITICAL'
        reason   = 'Model predicted critical + T3 confirms risk'
    elif pred == 'critical' and n_recovering >= 2:
        category = '🟠 RECOVERING'
        reason   = 'Model predicted critical but T3 shows improvement'
    elif pred == 'recovery' and n_at_risk_t3 >= 3:
        category = '🟠 DETERIORATING'
        reason   = 'Model predicted recovery but T3 shows deterioration'
    elif pred == 'recovery' and n_at_risk_t3 >= 1:
        category = '🟡 RECOVERY_ALERT'
        reason   = 'Model predicted recovery + T3 confirms 1-2 subjects at risk'
    elif pred == 'no_risk' and n_at_risk_t3 >= 2:
        category = '🟡 EARLY_WARNING'
        reason   = 'Model predicted no risk but T3 shows unexpected decline'
    elif pred in ['critical','recovery'] and conf < 0.50:
        category = '🔵 MONITOR'
        reason   = f'Uncertain prediction ({conf*100:.0f}% confidence) — watch closely'
    else:
        category = '🟢 ON_TRACK'
        reason   = 'No significant risk indicators'

    rec['alert_category'] = category
    rec['alert_reason']   = reason
    results.append(rec)

df_alerts = pd.DataFrame(results)

print("=" * 60)
print("EARLY ALERT SYSTEM V2 — Vermont School 2025-26")
print("=" * 60)
print(f"\nAlert distribution:")
print(df_alerts['alert_category'].value_counts().to_string())
print(f"\n── By grade ──")
print(df_alerts.groupby(['grade','alert_category'])\
    .size().unstack(fill_value=0).to_string())
print(f"\n── Subject projection summary ──")
print(f"Students with impossible subjects:  {(df_alerts['n_impossible']>0).sum()}")
print(f"Students with subjects at risk T3:  {(df_alerts['n_at_risk_t3']>0).sum()}")
print(f"Students recovering in T3:          {(df_alerts['n_recovering']>0).sum()}")

In [0]:
# CELDA 4 — Save alerts and generate dashboard visuals

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Save to Silver
spark.createDataFrame(df_alerts).write\
    .mode("overwrite").parquet(ALERT_OUTPUT)
print(f"✓ Alerts saved: {ALERT_OUTPUT}")

# Color map
colors = {
    '🔴 CONFIRMED_CRITICAL': '#e74c3c',
    '🟠 DETERIORATING':      '#e67e22',
    '🟠 RECOVERING':         '#f39c12',
    '🟡 RECOVERY_ALERT':     '#f1c40f',
    '🔵 MONITOR':            '#3498db',
    '🟢 ON_TRACK':           '#2ecc71',
}

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Vermont School 2025-26 — Early Warning System V2\nModel (24-25) + T3 Partial Reality',
             fontweight='bold', fontsize=14)

# ── Plot 1: Overall distribution (pie) ──
cats = df_alerts['alert_category'].value_counts()
axes[0,0].pie(
    cats.values,
    labels=[c[2:].replace('_',' ') for c in cats.index],
    colors=[colors[c] for c in cats.index],
    autopct='%1.1f%%',
    startangle=90,
    textprops={'fontsize': 9}
)
axes[0,0].set_title('Alert Distribution (149 students)', fontweight='bold')

# ── Plot 2: By grade (stacked bar) ──
grade_data = df_alerts.groupby(['grade','alert_category'])\
    .size().unstack(fill_value=0)
cat_order = ['🔴 CONFIRMED_CRITICAL','🟠 DETERIORATING','🟠 RECOVERING',
             '🟡 RECOVERY_ALERT','🔵 MONITOR','🟢 ON_TRACK']
grade_data = grade_data.reindex(
    columns=[c for c in cat_order if c in grade_data.columns],
    fill_value=0
)
bottom = np.zeros(len(grade_data))
for cat in grade_data.columns:
    axes[0,1].bar(
        [f'Grade {g}°' for g in grade_data.index],
        grade_data[cat].values,
        bottom=bottom,
        color=colors.get(cat, '#gray'),
        label=cat[2:].replace('_',' '),
        width=0.5
    )
    bottom += grade_data[cat].values
axes[0,1].set_title('Alert Distribution by Grade', fontweight='bold')
axes[0,1].set_ylabel('Students')
axes[0,1].legend(fontsize=7, loc='upper right')

# ── Plot 3: Confirmed critical — subject projection ──
df_critical = df_alerts[df_alerts['alert_category']=='🔴 CONFIRMED_CRITICAL'].copy()
subj_cols = [f'{g}_projected' for g in GROUPS if f'{g}_projected' in df_critical.columns]
df_proj = df_critical[['student_id'] + subj_cols].set_index('student_id')
df_proj.columns = [c.replace('_projected','').replace('_',' ') for c in df_proj.columns]

im = axes[1,0].imshow(df_proj.values, cmap='RdYlGn', vmin=1, vmax=7, aspect='auto')
axes[1,0].set_xticks(range(len(df_proj.columns)))
axes[1,0].set_xticklabels(df_proj.columns, rotation=45, ha='right', fontsize=7)
axes[1,0].set_yticks(range(len(df_proj.index)))
axes[1,0].set_yticklabels(df_proj.index, fontsize=7)
axes[1,0].axvline(x=-0.5, color='black', linewidth=0.5)
plt.colorbar(im, ax=axes[1,0], label='Projected grade')
axes[1,0].set_title('Confirmed Critical — Projected Grades by Subject\n(green=passing, red=failing)',
                    fontweight='bold')
# Add 4.0 threshold line overlay
for j in range(len(df_proj.columns)):
    for i in range(len(df_proj.index)):
        val = df_proj.values[i,j]
        if not np.isnan(val):
            color = 'white' if val < 4.5 else 'black'
            axes[1,0].text(j, i, f'{val:.1f}', ha='center', va='center',
                          fontsize=6, color=color)

# ── Plot 4: Model confidence distribution ──
conf_data = df_alerts.groupby('alert_category')['confidence'].mean().reindex(cat_order).dropna()
bars = axes[1,1].barh(
    [c[2:].replace('_',' ') for c in conf_data.index],
    conf_data.values,
    color=[colors[c] for c in conf_data.index]
)
axes[1,1].axvline(x=0.5, color='black', linestyle='--', 
                  linewidth=1.5, label='50% threshold')
axes[1,1].set_xlabel('Average model confidence')
axes[1,1].set_title('Average Model Confidence by Alert Category', fontweight='bold')
axes[1,1].legend(fontsize=8)
for bar, val in zip(bars, conf_data.values):
    axes[1,1].text(val + 0.01, bar.get_y() + bar.get_height()/2,
                  f'{val*100:.0f}%', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('/tmp/early_warning_dashboard_v2.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Dashboard saved")

print(f"\n{'='*60}")
print(f"NOTEBOOK 04 COMPLETE — Early Warning System V2")
print(f"{'='*60}")
print(f"\n  🔴 CONFIRMED CRITICAL: {(df_alerts['alert_category']=='🔴 CONFIRMED_CRITICAL').sum():3d}")
print(f"  🟠 DETERIORATING:      {(df_alerts['alert_category']=='🟠 DETERIORATING').sum():3d}")
print(f"  🟠 RECOVERING:         {(df_alerts['alert_category']=='🟠 RECOVERING').sum():3d}")
print(f"  🟡 RECOVERY ALERT:     {(df_alerts['alert_category']=='🟡 RECOVERY_ALERT').sum():3d}")
print(f"  🔵 MONITOR:            {(df_alerts['alert_category']=='🔵 MONITOR').sum():3d}")
print(f"  🟢 ON TRACK:           {(df_alerts['alert_category']=='🟢 ON_TRACK').sum():3d}")
print(f"\n  Model F1-Score (CV 24-25): 0.7541")
print(f"  Training: 117 students | Prediction: 149 students")
print(f"\n✓ Alerts saved: {ALERT_OUTPUT}")

In [0]:
# CELDA 5 — Anonymized alert report (for academic documentation)

ALERT_ORDER = [
    '🔴 CONFIRMED_CRITICAL',
    '🟠 DETERIORATING', 
    '🟠 RECOVERING',
    '🟡 RECOVERY_ALERT',
    '🔵 MONITOR',
    '🟢 ON_TRACK'
]

GROUPS = [
    'Science', 'I_and_S', 'Mathematics', 'English',
    'Lengua_Castellana', 'Mandarin', 'Financial_Maths',
    'ICT_STEM', 'Physical_Education', 'Research_Methodology'
]

print("=" * 70)
print("VERMONT SCHOOL 2025-26 — EARLY WARNING REPORT (ANONYMIZED)")
print("Model trained on 24-25 | T1+T2 features | T3 partial confirmation")
print("=" * 70)

# Show only students needing attention, ordered by alert category
df_attention = df_alerts[df_alerts['alert_category'] != '🟢 ON_TRACK']\
    .sort_values([
        'alert_category',
        'avg_cumulative'
    ], key=lambda x: x.map(
        {a: i for i, a in enumerate(ALERT_ORDER)}
    ) if x.name == 'alert_category' else x)

for seccion in sorted(df_alerts['section_anon'].unique()):
    df_sec = df_attention[df_attention['section_anon'] == seccion]
    if len(df_sec) == 0:
        continue

    total_sec = len(df_alerts[df_alerts['section_anon'] == seccion])
    print(f"\n{'='*70}")
    print(f"SECTION: {seccion} "
          f"({total_sec} students — {len(df_sec)} need attention)")
    print(f"{'='*70}")

    for _, row in df_sec.iterrows():
        print(f"\n  {row['alert_category']}")
        print(f"  Student ID:  {row['student_id']}")
        print(f"  Reason:      {row['alert_reason']}")
        print(f"  Confidence:  {row['confidence']*100:.0f}%")
        print(f"  Current avg: {row['avg_cumulative']:.2f} | "
              f"Min grade: {row['min_cumulative']:.2f} | "
              f"Subjects below 4.0: {row['n_subjects_below_4']}")

        # Subject projections — only at-risk subjects
        has_projections = False
        for g in GROUPS:
            proj      = row.get(f'{g}_projected', np.nan)
            t3_status = row.get(f'{g}_T3_status', None)
            min_t3    = row.get(f'{g}_min_T3', np.nan)
            t3_p      = row.get(f'{g}_T3_partial', np.nan)

            if pd.isna(proj):
                continue
            if t3_status in ['AT_RISK', 'IMPOSSIBLE'] or proj < 4.0:
                if not has_projections:
                    print(f"  Subject projections:")
                    has_projections = True
                icon = '❌' if t3_status == 'IMPOSSIBLE' else '⚠️'
                t3_str = f"T3 partial: {t3_p:.2f}" \
                         if not pd.isna(t3_p) else "T3: pending"
                print(f"    {icon} {g:25} "
                      f"Projected: {proj:.2f} | "
                      f"{t3_str} | "
                      f"Needs T3 ≥ {min_t3:.2f}")